# EFSM Phase 3 — QLoRA Fine-tuning (Google Colab)

**Runtime → Change runtime type → T4 GPU**

Before running, add these secrets (🔑 icon in left sidebar):
- `HF_TOKEN` — HuggingFace write token
- `WANDB_API_KEY` — Weights & Biases API key

In [ ]:
# Cell 1 — Secrets
from google.colab import userdata
import os

os.environ['HF_TOKEN']      = userdata.get('HF_TOKEN')
os.environ['WANDB_API_KEY'] = userdata.get('WANDB_API_KEY')

import wandb
wandb.login(key=os.environ['WANDB_API_KEY'])

from huggingface_hub import login
login(token=os.environ['HF_TOKEN'])

print('Authenticated.')

In [ ]:
# Cell 2 — Clone repo and install
import subprocess, os, sys

REPO_URL = 'https://github.com/tasbidrahman10/empathetic-voice-llm.git'
REPO_DIR = 'empathetic-voice-llm'

if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', REPO_URL], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)

os.chdir(REPO_DIR)
print('Working directory:', os.getcwd())

# Use sys.executable so pip installs into the same env as this notebook (CUDA torch intact)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt', '-q'], check=True)
print('Requirements installed.')
print('Torch CUDA available:', __import__('torch').cuda.is_available())

In [ ]:
# Cell 3 — Download JSONL data from HuggingFace Hub
from huggingface_hub import hf_hub_download
import os

os.makedirs('data', exist_ok=True)
CHECKPOINT_REPO = 'tasbid001/efsm-checkpoints'

for split in ['train', 'val', 'test']:
    dest = f'data/{split}.jsonl'
    if not os.path.exists(dest):
        hf_hub_download(
            repo_id=CHECKPOINT_REPO,
            filename=f'data/{split}.jsonl',
            repo_type='model',
            local_dir='.',
            token=os.environ['HF_TOKEN'],
        )
        print(f'Downloaded {dest}')
    else:
        print(f'{dest} already present')

for split in ['train', 'val', 'test']:
    count = sum(1 for _ in open(f'data/{split}.jsonl'))
    print(f'{split}: {count} conversations')

In [ ]:
# Cell 4 — Train (or resume from checkpoint)
import subprocess, os, sys

CHECKPOINT_REPO = 'tasbid001/efsm-checkpoints'
RESUME_CHECKPOINT = 'checkpoint-400'  # change to latest checkpoint number, or set to None to start fresh
LOCAL_CHECKPOINT = f'checkpoints/{RESUME_CHECKPOINT}'

# Download checkpoint from HF Hub if resuming and not already present locally
if RESUME_CHECKPOINT and not os.path.exists(LOCAL_CHECKPOINT):
    print(f'Downloading {RESUME_CHECKPOINT} from HF Hub...')
    from huggingface_hub import snapshot_download
    snapshot_download(
        repo_id=CHECKPOINT_REPO,
        repo_type='model',
        local_dir='checkpoints',
        allow_patterns=f'{RESUME_CHECKPOINT}/*',
        token=os.environ['HF_TOKEN'],
    )
    print('Checkpoint downloaded.')

env = os.environ.copy()
env['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'

cmd = [sys.executable, 'src/training/train.py', '--config', 'configs/config.yaml']
if RESUME_CHECKPOINT:
    cmd += ['--resume_from_checkpoint', LOCAL_CHECKPOINT]

result = subprocess.run(cmd, env=env, stderr=subprocess.STDOUT, stdout=subprocess.PIPE, text=True)
print(result.stdout)
if result.returncode != 0:
    raise RuntimeError(f"train.py failed with exit code {result.returncode}")
print('Training complete.')

In [ ]:
# Cell 5 — Verify checkpoint on HuggingFace Hub
from huggingface_hub import list_repo_files
import os

files = list(list_repo_files('tasbid001/efsm-checkpoints', token=os.environ['HF_TOKEN']))
print('Files on HF Hub:')
for f in sorted(files):
    print(' ', f)